# X-PrivScore — TIFS Extension Experiments (Colab Pro)

**قبل التشغيل:** Runtime ← Change runtime type ← **A100 GPU** (أو L4 إذا A100 مشغولة).
وفعّلي **Background execution** من إعدادات Colab Pro حتى تستمر الجلسة لو أغلقتي التبويب.

هذه التجارب الثلاث ترفع الورقة لمستوى TIFS:
1. **K-scaling**: K ∈ {10,20,50,100} — قابلية التوسّع.
2. **10 seeds** بدل 3 — فواصل ثقة أضيق.
3. **R-sweep + AMR scaling** — ربط النظرية بالتجربة.

In [ ]:
# ── 1) Drive + code + deps ──
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
BASE = '/content/drive/MyDrive/xprivscore'
if not os.path.exists(BASE):
    with zipfile.ZipFile('/content/drive/MyDrive/xprivscore.zip') as z:
        z.extractall('/content/drive/MyDrive/')
%cd {BASE}
!pip -q install opacus shap kaggle
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# ── 2) Data check (البيانات منزّلة من قبل) ──
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KEY'  # إذا احتجتي إعادة تنزيل
!ls -lh data/ | grep -E 'creditcard|elliptic_txs|train_trans' || echo 'البيانات مفقودة - شغّلي خلية التنزيل القديمة'

In [ ]:
# ── 3) AMR scaling study (CPU، دقائق) — أرخص تجربة وأقوى نتيجة ──
# تثبت: حد Lemma 1 يصمد لكل N، وAMR أسرع 55× عند N=100
!python amr_scaling_study.py

In [ ]:
# ── 4) K-scaling: الأولوية 1 (IEEE وElliptic حتى K=100، ULB حتى K=50) ──
# كل أمر = 3 seeds × 200 جولة × K عميل
for K in [20, 50]:
    !python run_experiment.py --dataset ulb --model mlp --n-clients {K} --name kscale_ulb_K{K}
for K in [20, 50, 100]:
    !python run_experiment.py --dataset elliptic --model mlp --n-clients {K} --name kscale_elliptic_K{K}
    !python run_experiment.py --dataset ieee --model mlp --n-clients {K} --name kscale_ieee_K{K}

In [ ]:
# ── 5) 10 seeds للنتائج الرئيسية (الأولوية 2) ──
for ds in ['ulb', 'elliptic', 'ieee']:
    !python run_experiment.py --dataset {ds} --model mlp --seeds 0 1 2 3 4 5 6 7 8 9 --name main10_{ds}

In [ ]:
# ── 6) k_trim sweep: ربط المتانة بالنظرية (الأولوية 3) ──
# يثبت Proposition 1: k_trim=2 يحمي من b=2 الذي كسر k_trim=1
for kt in [2, 3]:
    !python run_experiment.py --dataset ulb --model mlp --k-trim {kt} --attack scaling --n-malicious 2 --name ktrim{kt}_scaling_b2
    !python run_experiment.py --dataset ulb --model mlp --k-trim {kt} --name ktrim{kt}_benign

In [ ]:
# ── 7) تجميع كل النتائج ──
!zip -rq /content/drive/MyDrive/xprivscore_results_ext.zip results/
print('جاهز: MyDrive/xprivscore_results_ext.zip — أرسليه في المحادثة')